# Step 50 Run LOSO K-Sweep Model Selection

## What This Notebook Does

This is the first public Stage-5 notebook. It preserves the broad model-order screening workflow that:
- loads the canonical retained dataset from Stage 4
- runs a leave-one-subject-out K sweep over `K = 2..12`
- records validation and test free-energy summaries
- keeps anti-collapse diagnostics and feasibility metrics explicit
- writes the screening outputs used to decide which model orders are worth closer comparison

## When To Run It

Run this notebook after the canonical Stage-4 dataset is ready:
- `DATA_VARIANT = "intermediate"`
- `FEATURE_MODE = "nolags"`
- `MINLEN = 15`

Run it before:
- `step51_run_loso_shortlist_stability_checks.ipynb`
- `step52_build_figure2_and_table_s8_model_selection_summary.ipynb`

## Manuscript Linkage

- Main Methods 2.5.1
- Supplementary Methods 1.6-1.7
- Figure 2 support
- Table S8 support

## Inputs Expected

- the Stage-4 retained dataset root for the canonical manuscript branch
- `segments_manifest.tsv` from the canonical Stage-4 export, unless you provide an explicit override

## Outputs Written

- `cv_results.tsv`
- `cv_candidates_long.tsv`
- `fold_meta.tsv`
- `summary_byK_selected.tsv`
- `summary_byK_candidates.tsv`
- `paired_tests_vs_bestK.tsv`
- `K_selection_recommendation.json`
- screening plots such as `feasibility_vs_K.png` and `mean_testFE_with_SEM.png`

## Environment Notes

This notebook runs the actual TensorFlow and `osl_dynamics` screening workflow.
- it is GPU-sensitive in normal use
- CPU fallback is possible, but it is usually much slower and less practical for the full sweep
- chunk/resume controls are exposed near the top because longer runs may need multiple passes on a workstation or laptop

## Public Workflow Note

To keep this notebook readable, the dense training and cross-validation machinery now lives in a descriptive same-directory Python backend module that the public helper calls directly.

## Interpretation Note

This notebook is the broad screening step, not the final manuscript interpretation step.
It may still leave higher-K candidates such as `K=12` visible at the screening stage. The final manuscript choice is not made here by one automatic rule alone.


In [ ]:
# Step 0. Import Python modules and locate the active Stage-5 helper module

import sys
from pathlib import Path

STAGE5_DIR = Path.cwd()
candidate = Path.cwd() / "notebooks" / "5_hmm_selection"
if not (STAGE5_DIR / "stage5_hmm_selection_helpers.py").exists() and candidate.exists():
    STAGE5_DIR = candidate

if not (STAGE5_DIR / "stage5_hmm_selection_helpers.py").exists():
    raise FileNotFoundError(
        f"Stage-5 helper not found: {STAGE5_DIR / 'stage5_hmm_selection_helpers.py'}"
    )

if str(STAGE5_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(STAGE5_DIR.resolve()))

from stage5_hmm_selection_helpers import run_public_k_sweep_step


In [ ]:
# Step 1. User-editable roots and run settings
#
# SEGMENTS_ROOT should point to the Stage-4 retained-data root that contains
# the canonical `intermediate/` branch.
# MODEL_SELECTION_ROOT is where this notebook will write the broad K-sweep outputs.
# Leave MANIFEST_TSV as None unless you need to override the standard Stage-4
# manifest location on your machine.

SEGMENTS_ROOT = Path("<SET_SEGMENTS_ROOT>")
MODEL_SELECTION_ROOT = Path("<SET_MODEL_SELECTION_ROOT>")

DATA_VARIANT = "intermediate"
FEATURE_MODE = "nolags"
MINLEN = 15

# Optional explicit manifest override.
# Leave this as None to use the standard Stage-4 manifest location.
MANIFEST_TSV = None

# Broad screening grid for the manuscript model-order sweep.
K_GRID = list(range(2, 13))

# Chunk/resume and debug controls for long runs.
MAX_NEW_PAIRS_PER_RUN = 15
DEBUG_MAX_FOLDS = None
DEBUG_K_GRID = None
DEBUG_SEEDS = None

# Set to None to rely on TensorFlow memory-growth instead of a hard cap.
GPU_MEMORY_LIMIT_MB = 4096

FINAL_ROOT = SEGMENTS_ROOT / DATA_VARIANT
KSWEEP_OUTPUT_DIR = MODEL_SELECTION_ROOT / f"{DATA_VARIANT}_{FEATURE_MODE}_minlen{MINLEN}"


In [ ]:
# Step 2. Validate the configured inputs and print a short run summary

def ensure_configured_path(path_value, label, must_exist=False):
    path = Path(path_value)
    if "<SET_" in str(path):
        raise ValueError(f"{label} is still using a placeholder path. Edit this notebook before running.")
    if must_exist and not path.exists():
        raise FileNotFoundError(f"{label} does not exist: {path}")
    return path

SEGMENTS_ROOT = ensure_configured_path(SEGMENTS_ROOT, "SEGMENTS_ROOT", must_exist=True)
MODEL_SELECTION_ROOT = ensure_configured_path(MODEL_SELECTION_ROOT, "MODEL_SELECTION_ROOT")

print("Stage 5 / Step 50: broad LOSO K-sweep model selection")
print(f"  Canonical Stage-4 root: {FINAL_ROOT}")
print(f"  Output root:            {KSWEEP_OUTPUT_DIR}")
print(f"  Feature mode:           {FEATURE_MODE}")
print(f"  Minimum segment len:    {MINLEN} TR")
print(f"  K grid:                 {K_GRID}")
print(f"  GPU memory cap (MB):    {GPU_MEMORY_LIMIT_MB}")
print("  Packages needed:        tensorflow, osl_dynamics, numpy, pandas, matplotlib, scipy")
print("  Practical note:         CPU fallback is possible, but the full sweep is usually much slower without a GPU.")


## Step 3. Run the broad K-sweep screening workflow

The helper called below keeps the original screening behavior intact while making this public notebook easier to follow.

The key public idea is simple:
- Step 50 screens `K = 2..12`
- Step 51 compares the manuscript-facing shortlist
- Step 52 rebuilds the manuscript summary outputs


In [ ]:
# Step 3a. Run the active Python K-sweep backend through the public helper

run_result = run_public_k_sweep_step(
    segments_root=SEGMENTS_ROOT,
    model_selection_root=MODEL_SELECTION_ROOT,
    data_variant=DATA_VARIANT,
    feature_mode=FEATURE_MODE,
    minlen=MINLEN,
    manifest_tsv=MANIFEST_TSV,
    k_grid=K_GRID,
    max_new_pairs_per_run=MAX_NEW_PAIRS_PER_RUN,
    gpu_memory_limit_mb=GPU_MEMORY_LIMIT_MB,
    debug_max_folds=DEBUG_MAX_FOLDS,
    debug_k_grid=DEBUG_K_GRID,
    debug_seeds=DEBUG_SEEDS,
)

print("Using backend module:", run_result["backend_module"])
print("Backend status:", run_result["backend_status"])
if run_result["backend_message"]:
    print(run_result["backend_message"])
print("Resolved Stage-4 root:", run_result["final_root"])
print("Resolved manifest:", run_result["manifest_tsv"])
print("Screening output root:", run_result["out_root"])
run_result["available_outputs"]


In [ ]:
# Step 4. Review the key screening outputs that feed the shortlist stage

print("Available screening outputs:")
for label, path in run_result["available_outputs"].items():
    print(f"  {label}: {path}")

if run_result["summary_byk_selected_df"] is not None:
    print("\nSelected-model summary by K:")
    display(run_result["summary_byk_selected_df"])
else:
    print("\n`summary_byK_selected.tsv` is not available yet. This often means the run stopped after a chunk-resume boundary before the final summary files were rebuilt.")

if run_result["paired_tests_df"] is not None:
    print("\nPaired tests versus the current best K:")
    display(run_result["paired_tests_df"])

if run_result["recommendation"] is not None:
    print("\nCurrent K-selection recommendation:")
    run_result["recommendation"]
else:
    print("\n`K_selection_recommendation.json` is not available yet. If you are running in chunks, rerun the notebook until the screening summaries are complete.")
